# Phase 31 Step 311: Automated Entity Disambiguation

**Purpose**: Deterministic alignment bootstrap across sources for all core classes.

**Input**: Broadcasting programs, mention detection CSVs, Wikidata/Fernsehserien projections  
**Output**: aligned_*.csv files per core class in `data/31_entity_disambiguation/`

**Key Principles**:
- Layer 1-4 matching model (broadcasts, episodes, persons, roles/organizations)
- Precision-first: never force match below confidence threshold
- Event-sourced: all decisions logged for reproducibility and recovery
- One person mention in one episode = canonical matching unit

**Runtime**: Single `Run All` deterministic execution. No manual edits. Replayable from checkpoints.

In [ ]:
"""
Bootstrap: Discover repository root and set up import paths.
This cell runs first and must not depend on project modules.
"""
import sys
from pathlib import Path

# Find repo root by walking up from this notebook location
notebook_dir = Path.cwd()
repo_root = None

for parent in [notebook_dir] + list(notebook_dir.parents):
    if (parent / "speakermining" / "src" / "process").exists():
        repo_root = parent
        break

if not repo_root:
    raise RuntimeError(
        f"Could not find repository root from {notebook_dir}. "
        "Expected 'speakermining/src/process' folder structure."
    )

# Add src to path for process module imports
src_path = repo_root / "speakermining" / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(f"Repository root: {repo_root}")
print(f"Import path added: {src_path}")
print(f"Working directory: {Path.cwd()}")

In [ ]:
# Import project modules
import pandas as pd
from datetime import datetime, timezone

from process.entity_disambiguation import (
    Step311Orchestrator,
    RecoveryOrchestrator,
)
from process.entity_disambiguation.config import PHASE_DIR, get_aligned_csv_path, CORE_CLASSES

PROJECTION_ROOT = PHASE_DIR / "projections"

print("✓ All imports successful")

## Step 1: Check for Recovery Checkpoint

If this notebook was interrupted, detect and restore from the most recent checkpoint.

In [ ]:
# Check for recovery checkpoint
print("Checking for recovery checkpoint...")

import shutil

# Set to True to wipe local Step 311 state before running.
# This removes event-store chunks, checkpoints, handler progress DB,
# and the derived projections folder under data/31_entity_disambiguation.
RESET_STATE = True

# Set to True only when you want to bypass recovery loading entirely.
FORCE_FRESH_RUN = False

if RESET_STATE:
    print("✓ Clean-slate reset enabled; wiping Step 311 local state")

    events_dir = PHASE_DIR / "events"
    checkpoints_dir = PHASE_DIR / "checkpoints"
    handler_progress_db = PHASE_DIR / "handler_progress.db"
    projection_root = PHASE_DIR / "projections"

    removed = []

    if events_dir.exists():
        shutil.rmtree(events_dir)
        removed.append(str(events_dir))

    if checkpoints_dir.exists():
        shutil.rmtree(checkpoints_dir)
        removed.append(str(checkpoints_dir))

    if handler_progress_db.exists():
        handler_progress_db.unlink()
        removed.append(str(handler_progress_db))

    if projection_root.exists():
        shutil.rmtree(projection_root)
        removed.append(str(projection_root))

    if removed:
        print("  Removed artifacts:")
        for p in removed:
            print(f"  - {p}")
    else:
        print("  Nothing to remove (state already clean)")

if FORCE_FRESH_RUN or RESET_STATE:
    print("✓ Recovery restore disabled for this run")
    recovered_projections = None
else:
    recovery_orchestrator = RecoveryOrchestrator()
    recovered_projections = recovery_orchestrator.recover_and_resume()

if recovered_projections:
    print("✓ Recovery checkpoint found and validated")
    print(f"  Recovered projections available: {list(recovered_projections.keys())}")
    print("✓ Proceeding with a full fresh import pass to detect new CSV content")
else:
    print("✓ No recovery checkpoint used; starting fresh run")

## Step 2: Run Deterministic Alignment Orchestration

Execute the complete Step 311 workflow if not recovered:

In [ ]:
try:
    print("=" * 70)
    print("STARTING STEP 311 AUTOMATED DISAMBIGUATION ORCHESTRATION")
    print("=" * 70)
    
    orchestrator = Step311Orchestrator()
    projections_dict = orchestrator.run()
    run_source = "fresh_run_after_recovery" if recovered_projections is not None else "fresh_run"
    
    print("\n" + "=" * 70)
    print("ORCHESTRATION COMPLETE")
    print("=" * 70)
    
    print(f"\nRun source: {run_source}")
    print(f"Output directory: {PHASE_DIR}")
    print(f"Projections built: {list(projections_dict.keys())}")
    
except KeyboardInterrupt:
    print("\n⚠ Notebook interrupted by user")
    print("Checkpoint saved. Rerun to resume from checkpoint.")
    raise
except Exception as e:
    print(f"\n✗ Error during orchestration: {e}")
    import traceback
    traceback.print_exc()
    raise

## Step 3: Verify Output and Display Summary

Show output file sizes and row counts per core class:

In [ ]:
print("\nOUTPUT VERIFICATION")
print("=" * 70)

summary_data = []

for core_class in CORE_CLASSES:
    csv_path = get_aligned_csv_path(core_class)
    
    if csv_path.exists():
        df = projections_dict.get(core_class)
        if df is not None and not df.empty:
            size_mb = csv_path.stat().st_size / (1024 * 1024)
            summary_data.append({
                "Core Class": core_class,
                "Row Count": len(df),
                "Column Count": len(df.columns),
                "File Size (MB)": f"{size_mb:.2f}",
                "Status": "✓ Written"
            })
        else:
            summary_data.append({
                "Core Class": core_class,
                "Row Count": 0,
                "Column Count": 0,
                "File Size (MB)": "0.00",
                "Status": "⊘ Empty"
            })
    else:
        summary_data.append({
            "Core Class": core_class,
            "Row Count": "N/A",
            "Column Count": "N/A",
            "File Size (MB)": "N/A",
            "Status": "✗ Missing"
        })

summary_df = pd.DataFrame(summary_data)
display(summary_df)

print(f"\nNext Step: Step 312 Manual Reconciliation in OpenRefine")
print(f"See: documentation/31_entity_disambiguation/312_manual_reconciliation_specification.md")
print(f"\nRun completed at: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}")

## Step 4: Inspect Produced Projections

Load all projection artifacts from the projections folder and expose them as DataFrames.

In [ ]:
projection_files = {
    core_class: get_aligned_csv_path(core_class)
    for core_class in CORE_CLASSES
}

projections_loaded = {}

for core_class, path in projection_files.items():
    if path.exists():
        projections_loaded[core_class] = pd.read_csv(path)
    else:
        projections_loaded[core_class] = pd.DataFrame()

broadcasting_programs_df = projections_loaded["broadcasting_programs"]
series_df = projections_loaded["series"]
episodes_df = projections_loaded["episodes"]
persons_df = projections_loaded["persons"]
topics_df = projections_loaded["topics"]
roles_df = projections_loaded["roles"]
organizations_df = projections_loaded["organizations"]

print(f"Projection root: {PROJECTION_ROOT}")
for core_class in CORE_CLASSES:
    path = projection_files[core_class]
    rows = len(projections_loaded[core_class])
    print(f"{core_class}: {rows} rows ({path})")

### Broadcasting Programs Projection

In [ ]:
broadcasting_programs_df

### Series Projection

In [ ]:
series_df

### Episodes Projection

In [ ]:
episodes_df

### Persons Projection

In [ ]:
persons_df

### Topics Projection

In [ ]:
topics_df

### Roles Projection

In [ ]:
roles_df

### Organizations Projection

In [ ]:
organizations_df